# Distributed ViT on CIFAR-10, 2x T4Set the notebook accelerator to **GPU T4 x2** before running. With any othersetting the `kaggle` suite still executes but `world_size` stays 1 and there isno scaling number to report.This notebook produces the only rows in `RESULTS.md` that come from realmulti device hardware.

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total,driver_version --format=csvimport torchprint("torch", torch.__version__)print("cuda", torch.cuda.is_available(), "| devices", torch.cuda.device_count())print("bf16 supported:", torch.cuda.is_bf16_supported())   # False on T4 (sm_75)

## Install`--no-deps` matters. The Kaggle image already ships a torch built against itsCUDA driver. Letting pip resolve this package's dependencies can pull adifferent torch wheel on top of it and break CUDA entirely.

In [ ]:
!git clone -q https://github.com/USER/distributed-vit-cifar10.git 2>/dev/null || true%cd /kaggle/working/distributed-vit-cifar10!pip install -q --no-deps -e .

## DataOn Kaggle the canonical host is normally reachable, so this takes the tarballpath and torchvision verifies its md5. If it is blocked, the script falls backto the mirror and every run artifact records which source was used.

In [ ]:
!python scripts/fetch_cifar10.py --root data

## Correctness before speedRunning the suite first would be measuring a system nobody has checked. Thisproves the two process gradient equals the single process gradient over thesame effective batch.

In [ ]:
!pip install -q pytest!python -m pytest -q

## The sweepSeven configurations: 1 and 2 ranks, fp32 and amp, DDP and FSDP, with andwithout torch.compile, plus one gradient accumulation run.FSDP is not expected to win here. At 1.8M parameters there is almost nothing toshard and the extra collectives are pure overhead. Measuring that is the point:"we chose DDP because FSDP measured slower at this model size" is a far betterinterview answer than "we used DDP."

In [ ]:
!python -m dvit.sweep kaggle --continue-on-error

## Results

In [ ]:
!python -m dvit.reportimport json, globfor p in sorted(glob.glob("runs/t4-*.json")):    r = json.load(open(p))    print(f"{r['name']:<32} ws={r['world_size']} "          f"{r['median_images_per_sec']:>9,.0f} img/s  "          f"peak {r['peak_memory_mb']:>7,.0f} MB  "          f"top1 {r['best_test_acc']*100:>6.2f}%")

## Take the artifacts home`runs/*.json` is the whole record. Download it, drop it into `runs/` locally,rerun `python -m dvit.report`, and the dashboard shows the T4 rows next to thelaptop rows with the speedup computed only within each hardware group.

In [ ]:
import shutilshutil.make_archive("/kaggle/working/runs-t4", "zip", "runs")print("download /kaggle/working/runs-t4.zip, unzip into runs/ locally")